## **IMPORTS | USAGE | DATA**
***

In [263]:
# As I am learning, I completed all these steps in Excel first to visualize what I am doing in Python. 

import pandas as pd
import numpy as np
from scipy.stats import norm

In [264]:
transaction = pd.read_csv("../data/transaction_data.csv")

In [265]:
transaction.columns

Index(['sku_number', 'inventory_type', 'stocking_type', 'leadTime',
       'unit_price', 'Manufacturing Site', 'division_code', 'transaction_date',
       'Order-Quantity'],
      dtype='object')

In [266]:
transaction.head()

,sku_number,inventory_type,stocking_type,leadTime,unit_price,Manufacturing Site,division_code,transaction_date,Order-Quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,0514,2023-01-01,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58
4,BF7C4C4D,FG,MTO,28,998.995060,US-86,BCF8,2023-01-01,93


# `Dataframe Formatting`

#### As you load and inspect your transaction dataframe, you can observe that the column names do not follow a similar convention. Apply the changes necessary to ensure all column names follow the same convention.

In [267]:
pt_1_transaction = transaction_data.rename(columns={'leadTime': 'lead_time','Manufacturing Site': 'manufacturing_site', 'Order-Quantity': 'order_quantity'})
pt_1_transaction.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,0514,2023-01-01,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58
4,BF7C4C4D,FG,MTO,28,998.995060,US-86,BCF8,2023-01-01,93


# `Data Inspection`

#### Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them.  Please remember to provide a note on any oddities you have found and a rationale for your suggested correction. You can do so by including a text for the changes you have applied in a corresponding markdown shell.

In [268]:
pt_1_transaction.describe()

,lead_time,unit_price,order_quantity
count,400634.000000,400634.000000,400634.000000
mean,25.760976,800.308766,79.933211
std,6.465948,374.746323,41.930024
min,14.000000,-85.222467,-98.000000
25%,28.000000,982.549477,63.000000
50%,28.000000,998.073561,93.000000
75%,28.000000,1006.155114,109.000000
max,2800.000000,1023.638643,194.000000


## My approach to `Data Inspection`
I have two SKUs with negative values in the **unit_price** column: **2D43CB75** and **2D43CB75**. 
SKU 2D43CB75 appears 742 times, SKU EE77A077 appears 2,759 times. 

The recommendation here is that I drop the two instances for which I have two negative values without knowing what they mean, it probably has to do with a price adjustment or price item was sold at a loss, limiting losses. I am going to restrict column to only show values greater or equal to zero, ">=0"

I know that the **order_quantity** column has negative (-) values but I am choosing to keep them, it may indicate backorders.

**lead_time** is large, upon closer inspection I notice that 2800 is incorrect, it should be 28 like the rest. I will replace that value.

In [269]:
# part 1
pt_1_transaction = pt_1_transaction[pt_1_transaction['unit_price'] >= 0]
pt_1_transaction.describe()

,lead_time,unit_price,order_quantity
count,400632.000000,400632.000000,400632.000000
mean,25.761000,800.313098,79.933575
std,6.465936,374.742239,41.929793
min,14.000000,31.467547,-98.000000
25%,28.000000,982.549477,63.000000
50%,28.000000,998.073561,93.000000
75%,28.000000,1006.155114,109.000000
max,2800.000000,1023.638643,194.000000


In [270]:
# part 2
pt_1_transaction['lead_time'] = pt_1_transaction['lead_time'].replace(2800, 28)
pt_1_transaction.describe()

,lead_time,unit_price,order_quantity
count,400632.000000,400632.000000,400632.000000
mean,25.754081,800.313098,79.933575
std,4.753693,374.742239,41.929793
min,14.000000,31.467547,-98.000000
25%,28.000000,982.549477,63.000000
50%,28.000000,998.073561,93.000000
75%,28.000000,1006.155114,109.000000
max,28.000000,1023.638643,194.000000


# `Nan Values`

#### Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Once more, remember to provide a note on the NaN values you found and a rationale for your suggested action.

In [271]:
# data types by column
pt_1_transaction.dtypes

sku_number             object
inventory_type         object
stocking_type          object
lead_time               int64
unit_price            float64
manufacturing_site     object
division_code          object
transaction_date       object
order_quantity          int64
dtype: object

In [272]:
# number of empty values by column
print(pt_1_transaction.isnull().sum())

sku_number              1265
inventory_type          5912
stocking_type           6441
lead_time                  0
unit_price                 0
manufacturing_site    100069
division_code          23757
transaction_date           0
order_quantity             0
dtype: int64


## My approach to `Nan Values`

I care about **sku_number** constrained by **stocking_type** that is MTS. 

In [273]:
# filtering by stocking_type
pt_1_transaction = pt_1_transaction[(pt_1_transaction['stocking_type'] == "MTS")]
pt_1_transaction.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,0514,2023-01-01,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58
6,3D7DF103,FG,MTS,28,996.426246,PL-F5,3B87,2023-01-01,104


In [274]:
# dropping empty, null, nan values in sku_number column
pt_1_transaction = pt_1_transaction[pt_1_transaction['sku_number'].notna() & (pt_1_transaction['sku_number'] != "")]
pt_1_transaction.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
0,F544EBC2,WIP,MTS,28,993.199814,NaN,0514,2023-01-01,97
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58
6,3D7DF103,FG,MTS,28,996.426246,PL-F5,3B87,2023-01-01,104


# `Usesful Information`

#### It's time to focus on the subset of this dataset we need. Since we are ultimately interested in calculating safety stock, we would like to focus on finished goods that are designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).
#### •	How many unique SKUs are we working with?
#### •	How many unique Manufacturing Sites are we working with?
#### •	How many Divisions are we working with?
#### •	What are the top and bottom 10 transactions by order quantity?
#### •	What are the top and bottom 10 transactions by total sales value? Hint: Notice sales is not a feature of this dataset, thus we must calculate it using the existing features.

## My approach to `Useful Information`

I have already filtered the data in the previous step to only include MTS in **stocking_type** column

In [275]:
# distinct values in sku_number column
pt_1_transaction['sku_number'].describe()

count       247390
unique         392
top       A4D2AB46
freq          5507
Name: sku_number, dtype: object

In [276]:
# distinct values in manufacturing_site column
pt_1_transaction['manufacturing_site'].describe()

count     185531
unique        15
top        PL-F5
freq       61884
Name: manufacturing_site, dtype: object

In [277]:
# distinct values in division_code column
pt_1_transaction['division_code'].describe()

count     232735
unique        66
top         06CC
freq       14816
Name: division_code, dtype: object

In [278]:
# top 10 transactions by order_quantity (biggest)
pt_1_transaction = pt_1_transaction.sort_values(by='order_quantity', ascending=False).head(10)

In [279]:
# bottom 10 transactions by order_quantity (smallest)
pt_1_transaction = pt_1_transaction.sort_values(by='order_quantity', ascending=True).head(10)

In [280]:
# adding a custom column to the dataframe
pt_1_transaction['aggregate_sales'] = pt_1_transaction['unit_price'] * pt_1_transaction['order_quantity']
pt_1_transaction.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,aggregate_sales
258579,E6E01400,FG,MTS,28,1015.596869,NaN,E202,2024-04-16,180,182807.436454
276850,633E44A4,FG,MTS,28,1001.723540,PL-F5,EBDE,2024-05-19,180,180310.237116
305472,0336A033,FG,MTS,28,999.224537,JP-8C,EBDE,2024-07-10,181,180859.641232
350090,6374C42B,RM,MTS,28,1009.715112,PL-D8,2DEB,2024-09-30,182,183768.150411
351878,32843A68,FG,MTS,28,1007.025152,US-86,06CC,2024-10-03,182,183278.577584


In [281]:
# top 10 transactions by aggregate_sales (biggest)
pt_1_transaction = pt_1_transaction.sort_values(by='aggregate_sales', ascending=False).head(10)

In [282]:
# bottom 10 transactions by aggregate_sales (smallest)
pt_1_transaction = pt_1_transaction.sort_values(by='aggregate_sales', ascending=True).head(10)

# `Required Transformation`

#### Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

In [283]:
# combined all steps from part 1 into one shell but I am now using a new dataframe name to avoid confusion
pt_2_transaction = transaction.rename(columns={'leadTime': 'lead_time','Manufacturing Site': 'manufacturing_site', 'Order-Quantity': 'order_quantity'})
pt_2_transaction = pt_2_transaction[pt_2_transaction['unit_price'] >= 0]
pt_2_transaction['lead_time'] = pt_2_transaction['lead_time'].replace(2800, 28)
pt_2_transaction = pt_2_transaction[(pt_2_transaction['stocking_type'] == "MTS")]
pt_2_transaction = pt_2_transaction[pt_2_transaction['sku_number'].notna() & (pt_2_transaction['sku_number'] != "")]
pt_2_transaction['aggregate_sales'] = pt_2_transaction['unit_price'] * pt_2_transaction['order_quantity']
pt_2_transaction.head()

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,aggregate_sales
0,F544EBC2,WIP,MTS,28,993.199814,NaN,0514,2023-01-01,97,96340.381912
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95,96113.855618
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103,103531.345433
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58,57834.013637
6,3D7DF103,FG,MTS,28,996.426246,PL-F5,3B87,2023-01-01,104,103628.329580


## My approach to `Required Transformation`

In [284]:
# manual process instead of using SciPy
pt_2_ss = pt_2_transaction.groupby('sku_number').agg(
    qty_min=('order_quantity', 'min'),
    qty_max=('order_quantity', 'max'),
    qty_avg=('order_quantity', 'mean'),
    qty_median=('order_quantity', 'median'),
    qty_var=('order_quantity', 'var'),
    qty_std=('order_quantity', 'std'),
    avg_lead_time=('lead_time', 'mean')
).reset_index()
print(pt_2_ss)
pt_2_ss.head()

    sku_number  qty_min  qty_max     qty_avg  qty_median      qty_var  \
0     0019425F      -15      180   97.113636        99.0   596.725206   
1     00DCA10C      -22       28    5.223827         6.0    86.246822   
2     010BA6D0      -15       33    4.327869         5.0    94.590710   
3     01FE860A      -21       27    3.666667         4.0    97.078431   
4     0336A033      -18      181   92.014820        97.0  1066.085232   
..         ...      ...      ...         ...         ...          ...   
387   FCE44A71      -26       29    4.420290         4.0   111.100171   
388   FDC1A54E      -15       30    4.838235         4.0    89.242098   
389   FDFDECEC       33      165  100.070588       100.0   394.143184   
390   FE55EA7C      -24      163   93.559152        98.0   895.193374   
391   FED86632      -17       28    5.085227         4.0    98.695552   

       qty_std  avg_lead_time  
0    24.427960      27.659091  
1     9.286917      14.000000  
2     9.725776      14.0000

,sku_number,qty_min,qty_max,qty_avg,qty_median,qty_var,qty_std,avg_lead_time
0,0019425F,-15,180,97.113636,99.0,596.725206,24.427960,27.659091
1,00DCA10C,-22,28,5.223827,6.0,86.246822,9.286917,14.000000
2,010BA6D0,-15,33,4.327869,5.0,94.590710,9.725776,14.000000
3,01FE860A,-21,27,3.666667,4.0,97.078431,9.852839,14.000000
4,0336A033,-18,181,92.014820,97.0,1066.085232,32.650961,28.000000


# `Safety Stock Calculation`

#### Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the **ppf** function of **SciPy.Stat** module of the **SciPy** Links to an external site. package, or simply use the numeric values from any standard normal distribution function.

## My approach to `Safety Stock Calculation`
1. Use pt_2_transaction
2. Use "safety_stock_formula.png" in the images folder

In [285]:
# using scipy.stats.norm (75%, 90%, 95%)
z75 = norm.ppf(0.75)
z90 = norm.ppf(0.90)
z95 = norm.ppf(0.95)

# use loc=0; safety stock = z * std_demand * sqrt(lead_time)
pt_2_ss['ss_75'] = z75 * pt_2_ss['qty_std'] * np.sqrt(pt_2_ss['avg_lead_time'])
pt_2_ss['ss_90'] = z90 * pt_2_ss['qty_std'] * np.sqrt(pt_2_ss['avg_lead_time'])
pt_2_ss['ss_95'] = z95 * pt_2_ss['qty_std'] * np.sqrt(pt_2_ss['avg_lead_time'])

pt_2_ss[['sku_number', 'ss_75', 'ss_90', 'ss_95']].head()

,sku_number,ss_75,ss_90,ss_95
0,0019425F,86.652580,164.642605,211.316495
1,00DCA10C,23.437480,44.531944,57.156131
2,010BA6D0,24.545033,46.636328,59.857079
3,01FE860A,24.865703,47.245612,60.639086
4,0336A033,116.533377,221.417052,284.185710


# `Safety Stock Distribution`

#### Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.
#### •	Which SKU has the largest safety stock?
#### •	Which SKU has the smallest safety stock?
#### •	What is the average safety stock value across all SKUs?

In [286]:
# which SKU has the largest safety stock (95%)
pt_2_ss.loc[pt_2_ss['ss_95'].idxmax(), 'sku_number']

'3D7DF103'

In [287]:
# which SKU has the smallest safety stock (95%)
# I copy
pt_2_ss.loc[pt_2_ss['ss_95'].idxmin(), 'sku_number']

'9820DE8D'

In [288]:
# what is the average safety stock (95%) across all SKUs
pt_2_ss['ss_95'].mean()

np.float64(131.74164441578444)

# **REFERENCES**
***

I used DataWrangler, a Microsoft package that visualizes the data like in excel and allows me to apply changes.

For `Safety Stock Distribution` I google how to use SciPy and statistics, as I am reading I notice trends and observe reoccurring words. I then search the user manual of Panfas to uderstand the code format. After reading, I realized that I must have "loc" to use "idxmax"

Google find-
https://stackoverflow.com/questions/60699836/how-to-use-norm-ppf

Google find-
https://stackoverflow.com/questions/67526556/pandas-loc-multiple-conditions-before-idxmax

"loc"
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html

"idxmax"
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.idxmax.html

"idxmin"
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.idxmin.html

explains the purpose of "idxmax"
https://www.educative.io/answers/what-is-pandas-idxmax-method-in-python